In [ ]:
path_prefix = '../..'

import sys
sys.path.append(path_prefix)

import json
import torch
from torch.utils.data import DataLoader

from egh_vlm.hallu_dataset import load_hallu_dataset, hallu_collate_fn, split_stratified
from egh_vlm.hallu_ffn_detector import HalluFFNDetector, eval_ffn_detector
from egh_vlm.utils import load_phd_dataset

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{path_prefix}/data/phd/features/features_phd_base_full_context_qwen3_layer24.pt'
DETECTOR_PATH = f'{path_prefix}/data/phd/detectors/ffn_detector_phd_base_full_context_qwen3_layer24.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/phd_base_full_context_qwen3_layer24_scored.json'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
with open(f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
features = load_hallu_dataset(f'{path_prefix}/data/phd/prototype/features_full.pt')
hidden_size = 2048

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_hallu_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

hidden_size = features.embs[0].size(-1) if len(features) > 0 else 0
print(f'Hidden size of features: {hidden_size}')

#### Analysis

In [ ]:
detector = HalluFFNDetector(
    input_dim=2048, 
    hidden_dim=2048, 
    output_dim=1,
    device=DEVICE)
detector.load_state_dict(torch.load(DETECTOR_PATH, map_location=DEVICE))
detector.eval()

In [ ]:
train_dataset, test_dataset = split_stratified(
    features, train_ratio=0.7, random_state=42
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    collate_fn=hallu_collate_fn,
    shuffle=True
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=hallu_collate_fn,
    shuffle=True,
)

In [ ]:
result = eval_ffn_detector(detector, test_dataloader)
print(f'Evaluation: Accuracy={result["acc"]}, F1={result["f1"]}, PR-AUC={result["pr_auc"]}')

#### Construct Scored Dataset

In [ ]:
scored_dataset = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_dataloader):
        ids, embs, grads, labels = batch
        scores = detector(embs, grads).squeeze()
        
        for id, score in zip(ids, scores):
            data_entry = next((item for item in dataset if item['id'] == id), None)
            if data_entry is None:
                print(f'Warning: ID {id} not found in dataset')
                continue
            else:
                data_entry['detector_score'] = float(score)
                scored_dataset.append(data_entry)
print(f'Length of scored dataset: {len(scored_dataset)}')

In [ ]:
with open(OUTPUT_PATH, 'w') as f:
    json.dump(scored_dataset, f, indent=4)